In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    mean_absolute_percentage_error
)

Đọc dữ liệu

In [ ]:

df = pd.read_csv("zillow_final.csv")

print("Shape:", df.shape)
df.head()

Shape: (4468, 33)


,price,bed,bath,living,lot_sqft,lot_living,bed_bath,type_condo,type_manufactured,type_multi,...,risk_overall,risk_loss,risk_social,risk_resilience,risk_fire,risk_earthquake,risk_heat,dist_city,dist_airport,dist_coast
0,4980000.0,4.0,5.0,4126.0,4922.000,1.192923,0.8,False,False,False,...,80.718966,89.710407,9.209856,12.692809,73.920540,94.138632,8.354783,13.715619,17.888756,13.623171
1,1215000.0,3.0,2.0,1825.0,7840.800,4.296329,1.5,False,False,False,...,75.714982,83.604998,23.062252,12.692809,18.467649,89.615069,13.904381,39.379252,34.192540,22.666476
2,2629000.0,4.0,4.0,3019.0,43381.404,14.369461,1.0,False,False,False,...,92.322785,96.534514,9.209856,12.692809,98.160370,93.614213,13.158396,18.768005,19.185657,11.646300
3,400000.0,3.0,2.0,944.0,4268.880,4.522119,1.5,False,False,True,...,68.423055,43.952134,93.082983,12.692809,0.000000,93.975717,20.717146,9.473542,11.193127,18.586372
4,849000.0,2.0,2.0,1154.0,5002.000,4.334489,1.0,False,False,False,...,53.028195,54.870000,40.578096,12.692809,0.000000,87.909814,8.907717,37.911992,27.545766,35.124601


 Tách X và y

In [ ]:

df = df.dropna(subset=["price"])

X = df.drop(columns=["price"])
y = df["price"]

# Đổi True/False thành 0/1 nếu có
for col in X.select_dtypes(include=["bool"]).columns:
    X[col] = X[col].astype(int)

# Điền missing value nếu có
X = X.fillna(X.median())

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (4468, 32)
y shape: (4468,)


Chia train/test

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (3574, 32)
Test: (894, 32)


Hàm đánh giá model

In [ ]:

def evaluate_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    median_ae = median_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    print("=" * 60)
    print(model_name)
    print("=" * 60)
    print(f"MAE       : {mae:,.0f}")
    print(f"Median AE : {median_ae:,.0f}")
    print(f"RMSE      : {rmse:,.0f}")
    print(f"R2        : {r2:.4f}")
    print(f"MAPE      : {mape:.2f}%")
    print()

ElasticNet = Linear + L1 + L2

In [ ]:

elastic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.001,      # độ phạt tổng thể
        l1_ratio=0.5,    # 0.5 nghĩa là L1 và L2 cân bằng nhau
        max_iter=50000,
        random_state=42
    ))
])

# Dùng log(price) vì giá nhà bị lệch mạnh
y_train_log = np.log1p(y_train)

elastic_model.fit(X_train, y_train_log)

# Dự đoán log(price)
y_pred_log = elastic_model.predict(X_test)

# Đổi ngược về giá thật
y_pred_elastic = np.expm1(y_pred_log)

evaluate_model(
    "ElasticNet Linear + L1 + L2 - Log Price",
    y_test,
    y_pred_elastic
)

ElasticNet Linear + L1 + L2 - Log Price
MAE       : 1,162,011
Median AE : 219,275
RMSE      : 5,197,972
R2        : 0.5168
MAPE      : 33.03%



Bảng kết quả

In [ ]:
elastic_results = pd.DataFrame([{
    "Model": "ElasticNet Linear + L1 + L2",
    "MAE": mean_absolute_error(y_test, y_pred_elastic),
    "Median AE": median_absolute_error(y_test, y_pred_elastic),
    "RMSE": mean_squared_error(y_test, y_pred_elastic) ** 0.5,
    "R2": r2_score(y_test, y_pred_elastic),
    "MAPE (%)": mean_absolute_percentage_error(y_test, y_pred_elastic) * 100
}])

elastic_results

,Model,MAE,Median AE,RMSE,R2,MAPE (%)
0,ElasticNet Linear + L1 + L2,1.162011e+06,219274.671842,5.197972e+06,0.51682,33.027848


Coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": elastic_model.named_steps["model"].coef_
})

coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

coef_df = coef_df.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

coef_df.head(15)

,Feature,Coefficient,Abs_Coefficient
11,lat,-0.522930,0.522930
12,long,-0.410169,0.410169
19,area_home_value,0.363447,0.363447
2,living,0.300950,0.300950
31,dist_coast,0.258351,0.258351
28,risk_heat,-0.218952,0.218952
9,type_single,0.175617,0.175617
30,dist_airport,-0.167327,0.167327
1,bath,0.135250,0.135250
7,type_manufactured,-0.125831,0.125831
